In this short notebook we inspect the data set of LLNAs and their performance regarding network classification. The dataset is taken from Lucas's research. We will concretely do the following:
1. Load all data per netwerk dataset
2. Visually inspect a number of well-performing rules
3. Trace genotype patterns within the subset of well-performing rules
4. For R=3 and R=5, make spacetime patterns on a ring network and look at common behaviour
5. Trace phenotype patterns within the spacetime patterns of well-performing rules
6. Request R=9 examples of "good rules", and inspect behaviour in a grid (using Golly)

# Load and explore all data

In [ ]:
# load tab-separated data
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import igraph as ig

# progress bar
from tqdm import tqdm

# animations
import matplotlib.animation as animation
from IPython.display import HTML

# Enable LaTeX and set Times New Roman as the font
from matplotlib import rcParams
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

from src.automata import LLNA
from src.simulation import *

%load_ext autoreload
%autoreload 2

# Load the TSV file into a DataFrame
data_dir = "../data/temp/"
data_name = "accuracy-vs-jaggedness.tsv"
df = pd.read_csv(data_dir + data_name, sep="\t")

# Display the DataFrame
df.head(3)

In [ ]:
# add binary sequence column (for later)

def extract_interval_indices(llna):
    parts = llna.split('_') # split along underscore
    first_part = parts[0][1:]  # Remove the 'B'
    first_list = [int(x) for x in first_part.split(',') if x]  # Split by commas and convert to integers
    second_part = parts[1][1:]  # Remove the 'S'
    second_list = [int(x) for x in second_part.split(',') if x]  # Convert to integer and put it in a list
    return first_list, second_list

def binary_sequence_from_rule(rule, resolution):
    born_if, survive_if = extract_interval_indices(rule)
    born_bin_seq = [1 if index in born_if else 0 for index in range(resolution)]
    surv_bin_seq = [1 if index in survive_if else 0 for index in range(resolution)]
    return born_bin_seq, surv_bin_seq

df['binary sequence B'] = df.apply(lambda row: binary_sequence_from_rule(row['rule'], row['resolution'])[0], axis=1)
df['binary sequence S'] = df.apply(lambda row: binary_sequence_from_rule(row['rule'], row['resolution'])[1], axis=1)

df.head(3)

Add some more metrics to the Dataframe

In [ ]:
from scipy.stats import entropy
from itertools import groupby
from sklearn.metrics import mutual_info_score

def shannon_entropy(sequence):
    # Count the frequency of 1s and 0s
    counts = np.bincount(sequence)
    # Normalize the counts to get probabilities
    probs = counts / len(sequence)
    return entropy(probs, base=2)

def hamming_weight_difference(sequence):
    sequence = (np.array(sequence)-0.5)*2
    hwd = np.sum(sequence) / len(sequence)
    return hwd

def average_run_length(sequence):
    # NOTE: the average run length of a random sequence is 2
    # Get runs of consecutive identical values
    runs = [len(list(group)) for key, group in groupby(sequence)]
    # Compute the average length of runs
    return sum(runs) / len(runs)

def hamming_distance(seq1, seq2):
    return np.sum(seq1 != seq2)

def autocorrelation(sequence):
    return np.corrcoef(sequence[:-1], sequence[1:])[0, 1]

def lempel_ziv_complexity(sequence):
    sequence_str = ''.join(map(str, sequence))
    substrings = set()
    i = 0
    while i < len(sequence_str):
        for j in range(i+1, len(sequence_str)+1):
            substring = sequence_str[i:j]
            if substring not in substrings:
                substrings.add(substring)
                i = j - 1
                break
        i += 1
    return len(substrings)

def fano_factor(sequence, window_size=5):
    # NOTE: makes little sense, the sliding window is not very interesting
    variances = []
    for i in range(len(sequence) - window_size + 1):
        window = sequence[i:i + window_size]
        variances.append(np.var(window))
    return np.mean(variances) / np.mean(variances)

def mutual_information(seq1, seq2):
    return mutual_info_score(seq1, seq2)

print('shannon (takes a while)')
df['shannon entropy B'] = df.apply(lambda row: shannon_entropy(row['binary sequence B']), axis=1)
df['shannon entropy S'] = df.apply(lambda row: shannon_entropy(row['binary sequence S']), axis=1)

print('hwd')
df['hamming weight difference B'] = df.apply(lambda row: hamming_weight_difference(row['binary sequence B']), axis=1)
df['hamming weight difference S'] = df.apply(lambda row: hamming_weight_difference(row['binary sequence S']), axis=1)

print('arl')
df['average run length B'] = df.apply(lambda row: average_run_length(row['binary sequence B']), axis=1)
df['average run length S'] = df.apply(lambda row: average_run_length(row['binary sequence S']), axis=1)

print('hamming d')
df['hamming distance'] = df.apply(lambda row: hamming_distance(row['binary sequence B'], row['binary sequence S']), axis=1)

print('autocrr')
df['autocorrelation B'] = df.apply(lambda row: autocorrelation(row['binary sequence B']), axis=1)
df['autocorrelation S'] = df.apply(lambda row: autocorrelation(row['binary sequence S']), axis=1)

print('lzc')
df['lempel ziv complexity B'] = df.apply(lambda row: lempel_ziv_complexity(row['binary sequence B']), axis=1)
df['lempel ziv complexity S'] = df.apply(lambda row: lempel_ziv_complexity(row['binary sequence S']), axis=1)

print('mi (takes a while)')
df['mutual information'] = df.apply(lambda row: mutual_information(row['binary sequence B'], row['binary sequence S']), axis=1)

In [ ]:
### Explore the accuracy spread over the various datasets

# Calculate the median accuracy for each dataset
median_accuracies = df.groupby('dataset')['accuracy'].median().sort_values()

# Reorder the DataFrame based on the sorted order of datasets
df['dataset'] = pd.Categorical(df['dataset'], categories=median_accuracies.index, ordered=True)

# 3. Create a boxplot, ordered by median accuracy
fontsize=16
fig, axs = plt.subplots(1, 3, figsize=(10, 6), sharey=True)
flierprops = dict(marker='o', markersize=3)  # Change markersize here

for resolution, ax in zip([3, 4, 5], axs):
    df_resolution = df[df["resolution"]==resolution]
    df_resolution.boxplot(ax=ax, column='accuracy', by='dataset', patch_artist=True,
              medianprops={'color': 'red'},
              flierprops=flierprops,
              vert=False, grid=True)
    ax.set_title(f"Resolution {resolution}")
    ax.set_ylabel(None)
    ax.set_xlim([0,1])

fig.supxlabel("Accuracy", fontsize=fontsize)
fig.supylabel("Dataset", fontsize=fontsize)
fig.suptitle("Boxplots of network classification accuracies per LLNA resolution", fontsize=fontsize)

fig.tight_layout()

# Select a relevant dataset and plot some genotype diagrams

I think it is interesting to look at a dataset for which the accuracy spread is quite large. One such set is the actinobacteria dataset. For each resolution, let's inspect the six best LLNAs, and try to see a pattern in the genotype.

No such pattern is immediately visible. We will further investigate binary patterns later on, however.

In [ ]:
# select dataset
dataset = 'actinobacteria'
df_dataset = df[df['dataset']==dataset]

# select resolution and sort
df_dataset_R3 = df_dataset[df_dataset["resolution"]==3].sort_values(by='accuracy', ascending=False).reset_index(drop=True)
df_dataset_R4 = df_dataset[df_dataset["resolution"]==4].sort_values(by='accuracy', ascending=False).reset_index(drop=True)
df_dataset_R5 = df_dataset[df_dataset["resolution"]==5].sort_values(by='accuracy', ascending=False).reset_index(drop=True)

# find five best LLNAs per resolution
select_first_somany = 6
llna_list_R3 = df_dataset_R3['rule'].to_numpy()[:select_first_somany]
llna_list_R4 = df_dataset_R4['rule'].to_numpy()[:select_first_somany]
llna_list_R5 = df_dataset_R5['rule'].to_numpy()[:select_first_somany]

In [ ]:
fontsize=16

fig, axs = plt.subplots(select_first_somany,1,figsize=(10,6*select_first_somany/5), sharex=True)

resolution=3
for llna, ax in zip(llna_list_R3, axs):
    born_if, survive_if = extract_interval_indices(llna)
    model = LLNA(resolution, x=born_if, y=survive_if)
    model.diagram(ax=ax)
    ax.set_ylabel(None)
    ax.set_xlabel(None)
    ax.get_legend().remove()
ax.legend(ncols=2, loc='upper right')

fig.suptitle(f"The top {select_first_somany} highest-accuracy LLNAs for dataset '{dataset}' and resolution {resolution}.", fontsize=fontsize)
fig.supxlabel(fr"State density $\rho$ of neighbourhood", fontsize=fontsize)
fig.supylabel(f"Node becomes ...", fontsize=fontsize)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(select_first_somany,1,figsize=(10,6*select_first_somany/5), sharex=True)

resolution=4
for llna, ax in zip(llna_list_R4, axs):
    born_if, survive_if = extract_interval_indices(llna)
    model = LLNA(resolution, x=born_if, y=survive_if)
    model.diagram(ax=ax)
    ax.set_ylabel(None)
    ax.set_xlabel(None)
    ax.get_legend().remove()
ax.legend(ncols=2, loc='upper right')

fig.suptitle(f"The top {select_first_somany} highest-accuracy LLNAs for dataset '{dataset}' and resolution {resolution}.", fontsize=fontsize)
fig.supxlabel(fr"State density $\rho$ of neighbourhood", fontsize=fontsize)
fig.supylabel(f"Node becomes ...", fontsize=fontsize)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(select_first_somany,1,figsize=(10,6*select_first_somany/5), sharex=True)

resolution=5
for llna, ax in zip(llna_list_R5, axs):
    born_if, survive_if = extract_interval_indices(llna)
    model = LLNA(resolution, x=born_if, y=survive_if)
    model.diagram(ax=ax)
    ax.set_ylabel(None)
    ax.set_xlabel(None)
    ax.get_legend().remove()
ax.legend(ncols=2, loc='upper right')

fig.suptitle(f"The top {select_first_somany} highest-accuracy LLNAs for dataset '{dataset}' and resolution {resolution}.", fontsize=fontsize)
fig.supxlabel(fr"State density $\rho$ of neighbourhood", fontsize=fontsize)
fig.supylabel(f"Node becomes ...", fontsize=fontsize)
fig.tight_layout()

# Show correlation with some binary sequence metrics

We will consider a number of binary sequence metrics that are well-defined for short binary sequences
1. Shannon entropy
2. Hamming weight (we will take the difference between #1 and #0)
3. Run length (related to jaggedness)
4. Autocorrelation
5. Lempel-Ziv complexity
6. Binary sequence distribution (frequency of 0s and 1s)
7. Fano factor

Note that these are most probably highly correlated

In [ ]:
df_dataset_R5.columns

In [ ]:
metric = 'mutual information'
df_dataset_R5.plot(kind='scatter', x=metric, y='accuracy', title=f'{metric} vs. Accuracy', alpha=.1)

Note: jaggedness appears to be perhaps the most informative of these measures!

# Spacetime patterns of these LLNAs on ring networks

Next, we run these LLNAs on a ring graph. Hopefully we can identify some common dynamical property. Afterwards we can also run it through the original network, but analysis on a non-regular topology is more ambiguous.

We work with undirected graphs, so we cannot investigate the $R=4$ rule in this case (which is perhaps another reason to drop even-resolution rules).

In [ ]:
# make a regular graph (emulating ECAs)
multiplier = 2
N = 2*3*4*multiplier+1

# ring graph
G_ring = ig.Graph.Ring(N)

# regular ringy graph
G_regular = ig.Graph(N)
# Add undirected edges to form a ring (nearest neighbors) and add second-nearest neighbors
G_regular.add_edges([(i, (i + 1) % N) for i in range(N)])  # First neighbours
G_regular.add_edges([(i, (i + 2) % N) for i in range(N)])  # Second neighours

# get ID of edges (bidirectional)
G_ring.to_directed()
edges_ring = tc.tensor(G_ring.get_edgelist()).T
G_ring.to_undirected()

G_regular.to_directed()
edges_regular = tc.tensor(G_regular.get_edgelist()).T
G_regular.to_undirected()

# plot both just to be sure
fig, axs = plt.subplots(2,2,figsize=(6,6))

# plot topologies
vertex_size=10
ig.plot(G_ring, target=axs[0,0], vertex_size=vertex_size)
ig.plot(G_regular, target=axs[1,0], vertex_size=vertex_size)
# plot adjacency matrices
axs[0,1].imshow(np.array(G_ring.get_adjacency()), cmap='Greys')
axs[1,1].imshow(np.array(G_regular.get_adjacency()), cmap='Greys')

axs[0,0].set_title(fr"Topology ($|\mathcal{{N}}|=2, N={N}$)")
axs[0,1].set_title(fr"Adjacency matrix ($|\mathcal{{N}}|=2, N={N}$)")
axs[1,0].set_title(fr"Topology ($|\mathcal{{N}}|=4, N={N}$)")
axs[1,1].set_title(fr"Adjacency matrix ($|\mathcal{{N}}|=4, N={N}$)")

fig.tight_layout()

In [ ]:
resolution=3
spacetime_diagrams=[]
models = []
T=N//2
for llna in llna_list_R3:
    # create LLNA
    born_if, survive_if = extract_interval_indices(llna)
    model = LLNA(resolution, x=born_if, y=survive_if)
    # run model over N//2 timesteps (avoid resonance)
    initial_config = np.random.randint(0,2,N)
    H = model.forward(edges_ring, tc.tensor(initial_config[np.newaxis,:]), T=T)
    spacetime_diagram = np.array(H, dtype=int)[0]
    spacetime_diagrams.append(spacetime_diagram)
    models.append(model)

fig, axs = plt.subplots(2,3,figsize=(12,5))
for i in range(2):
    ax_row = axs[i]
    for ax, j in zip(ax_row, range(3)):
        index = i*3+j
        ax.set_title(model.__str__(latex=True), fontsize=16)
        ax.imshow(spacetime_diagrams[index], cmap='Greys')
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Spacetime diagrams for high-accuracy LLNAs (for dataset '{dataset}') on a ring network ($R={resolution}$).", fontsize=16)

# fig.tight_layout()

In [ ]:
resolution=5
spacetime_diagrams=[]
models = []
T=N//2
for llna in llna_list_R5:
    # create LLNA
    born_if, survive_if = extract_interval_indices(llna)
    model = LLNA(resolution, x=born_if, y=survive_if)
    # run model over N//2 timesteps (avoid resonance)
    initial_config = np.random.randint(0,2,N)
    H = model.forward(edges_regular, tc.tensor(initial_config[np.newaxis,:]), T=T)
    spacetime_diagram = np.array(H, dtype=int)[0]
    spacetime_diagrams.append(spacetime_diagram)
    models.append(model)

fig, axs = plt.subplots(2,3,figsize=(12,5))
for i in range(2):
    ax_row = axs[i]
    for ax, j in zip(ax_row, range(3)):
        index = i*3+j
        ax.set_title(model.__str__(latex=True), fontsize=16)
        ax.imshow(spacetime_diagrams[index], cmap='Greys')
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Spacetime diagrams for high-accuracy LLNAs (for dataset '{dataset}') on a ring network ($R={resolution}$).", fontsize=16)

# fig.tight_layout()

# NOTE: here it would also be very interesting to have some kind of self-supervised classification based on simple computer vision techniques.

# Spacetime patterns of $R=5$ LLNAs on regular "two-dimensional" networks

It's not really clear what pattern I should be able to see above. Where is the "blinkedness" that Odemir spoke of? I could run this in another regular network though: a cellular automaton with the von Neumann neighbourhood.

In [ ]:
# make a regular graph (emulating a 2D von Neumann CA)
N = 50

# Create a toroidal 2D lattice with Von Neumann neighborhood (4 neighbors)
G_lattice = ig.Graph.Lattice([N, N], circular=True, nei=1)

# get ID of edges (bidirectional)
G_lattice.to_directed()
edges_lattice = tc.tensor(G_lattice.get_edgelist()).T
G_lattice.to_undirected()

# plot both just to be sure
fig, ax = plt.subplots(1,1,figsize=(5,5))

# plot topologies
vertex_size=5
ig.plot(G_lattice, target=ax, vertex_size=vertex_size)
ax.set_title(fr"Topology ($|\mathcal{{N}}|=4, N={N**2}$)")

fig.tight_layout()

In [ ]:
resolution=5
spacetime_diagrams=[]
models = []
T=N
for llna in llna_list_R5:
    # create LLNA
    born_if, survive_if = extract_interval_indices(llna)
    model = LLNA(resolution, x=born_if, y=survive_if)
    # run model over N//2 timesteps (avoid resonance)
    initial_config = np.random.randint(0,2,N**2)
    # initial_config = np.zeros(N**2)
    # initial_config[N**2//2+N//2] = 1
    H = model.forward(edges_lattice, tc.tensor(initial_config[np.newaxis,:]), T=T)
    spacetime_diagram = np.array(H, dtype=int)[0].reshape(T+1,N,N)
    spacetime_diagrams.append(spacetime_diagram)
    models.append(model)

def make_animation_object(spacetime_diagram, model):
    # Set up figure
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(spacetime_diagram[0], cmap='Greys', interpolation='nearest')
    # Update function for animation
    def update(frame):
        im.set_array(spacetime_diagram[frame])
        ax.set_title(f"{model.__str__(latex=True)} on a lattice network,\nTime step {frame}", fontsize=16)
        ax.set_xticks([]); ax.set_yticks([])
    # Create animation
    ani = animation.FuncAnimation(fig, update, frames=T+1, interval=200)
    plt.close()
    return ani

In [ ]:
SAVEGIF = True
example_index = 0

diagram = spacetime_diagrams[example_index]
model = models[example_index]

ani = make_animation_object(diagram, model)

if SAVEGIF:
    # Save animations as GIFs
    loc = '../figures/gifs/'
    savename = f"animation_{model.__str__()}_on_lattice.gif"
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF = True
example_index = 1

diagram = spacetime_diagrams[example_index]
model = models[example_index]

ani = make_animation_object(diagram, model)

if SAVEGIF:
    # Save animations as GIFs
    loc = '../figures/gifs/'
    savename = f"animation_{model.__str__()}_on_lattice.gif"
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF = True
example_index = 2

diagram = spacetime_diagrams[example_index]
model = models[example_index]

ani = make_animation_object(diagram, model)

if SAVEGIF:
    # Save animations as GIFs
    loc = '../figures/gifs/'
    savename = f"animation_{model.__str__()}_on_lattice.gif"
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF = True
example_index = 3

diagram = spacetime_diagrams[example_index]
model = models[example_index]

ani = make_animation_object(diagram, model)

if SAVEGIF:
    # Save animations as GIFs
    loc = '../figures/gifs/'
    savename = f"animation_{model.__str__()}_on_lattice.gif"
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF = True
example_index = 4

diagram = spacetime_diagrams[example_index]
model = models[example_index]

ani = make_animation_object(diagram, model)

if SAVEGIF:
    # Save animations as GIFs
    loc = '../figures/gifs/'
    savename = f"animation_{model.__str__()}_on_lattice.gif"
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF = True
example_index = 5

diagram = spacetime_diagrams[example_index]
model = models[example_index]

ani = make_animation_object(diagram, model)

if SAVEGIF:
    # Save animations as GIFs
    loc = '../figures/gifs/'
    savename = f"animation_{model.__str__()}_on_lattice.gif"
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

It's not clear to me which property we should call 'blinkedness' here.

# Best-performance $R=9$ LLNA rules in Life-Like CA

In [Zielinski et al. (2024)](https://www.sciencedirect.com/science/article/pii/S0031320323006441) section 4.2 we find a summary of some best-performance resolution-9 LLNAs. In the "old" notation, these are
- $B135678/S03456$ for the 4-models synthetic dataset and noisy-synthetic dataset
- $B01678/S0457$ for the 4-models + $\langle k \rangle$ synthetic dataset
- $B0157/S457$ for the scalefree synthetic dataset
- $B0167/S246$ for the  social dataset
- $B02345678/S123468$ for the kingdom dataset
- $B023468/S01468$  for the animal dataset
- $B04/S1468$ for the fungi dataset
- $B0468/S0467$ for the plant dataset
- $B0236/S123567$ for the protist dataset
- $B0468/S0458$ for the firmicutes-bacillis dataset
- $B1237/S267$ for the actibacteria dataset

Below we inspect their behaviour on a lattice network.

In [ ]:
from src.analysis import id_sensitivity, nbh_sensitivity

datasets = ['4-models synthetic dataset and noisy-synthetic',
            '4-models + <k> synthetic',
            'scalefree synthetic',
            'social',
            'kingdom',
            'animal',
            'fungi',
            'plant',
            'protist',
            'firmicutes-bacillis',
            'actibacteria']

intervals = [([1, 3, 5, 6, 7, 8], [0, 3, 4, 5, 6]),
             ([0, 1, 6, 7, 8], [0, 4, 5, 7]),
             ([0, 1, 5, 7], [4, 5, 7]),
             ([0, 1, 6, 7], [2, 4, 6]),
             ([0, 2, 3, 4, 5, 6, 7, 8], [1, 2, 3, 4, 6, 8]),
             ([0, 2, 3, 4, 6, 8], [0, 1, 4, 6, 8]),
             ([0, 4], [1, 4, 6, 8]),
             ([0, 4, 6, 8], [0, 4, 6, 7]),
             ([0, 2, 3, 6], [1, 2, 3, 5, 6, 7]),
             ([0, 4, 6, 8], [0, 4, 5, 8]),
             ([1, 2, 3, 7], [2, 6, 7])]

resolution = 9
models = [LLNA(resolution, x=interval[0], y=interval[1]) for interval in intervals]

IS_list = [id_sensitivity(resolution, interval[0], interval[1]) for interval in intervals]
NS_list = [nbh_sensitivity(resolution, interval[0], interval[1]) for interval in intervals]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,3))
ax.hist([IS_list, NS_list], label=["Identity Sensitivity", "Neighbourhood Sensitivity"], bins=6)
ax.legend(loc='upper right')
ax.set_xlabel("Sensitivity value")
ax.set_ylabel("Prevalence")
ax.set_title(fr"Sensitivity values of all $11$ best-performance LLNAs")
ax.set_xlim([0,1])

fig.tight_layout()

In [ ]:
# make a regular graph, emulating a Life-like CA (a 2D Moore CA)
N = 50

# Create a toroidal 2D lattice with Von Neumann neighborhood (4 neighbors)
G_lattice_moore = ig.Graph.Lattice([N, N], circular=True, nei=1)

# Function to get 1D index from (x, y) position in the NxN grid
def node_index(x, y, N):
    return (x % N) * N + (y % N)  # Apply modulo for periodic boundaries

# Manually add diagonal edges
for x in range(N):
    for y in range(N):
        node = node_index(x, y, N)
        
        # Diagonal neighbors
        diag_neighbors = [
            (x+1, y+1), (x-1, y-1),  # Bottom-right, Top-left
            (x+1, y-1), (x-1, y+1)   # Top-right, Bottom-left
        ]
        
        for xn, yn in diag_neighbors:
            neighbor = node_index(xn, yn, N)
            G_lattice_moore.add_edge(node, neighbor)

# get ID of edges (bidirectional)
G_lattice_moore.to_directed()
edges_lattice_moore = tc.tensor(G_lattice_moore.get_edgelist()).T
G_lattice_moore.to_undirected()

# plot both just to be sure
fig, ax = plt.subplots(1,1,figsize=(5,5))

# plot topologies
vertex_size=5
ig.plot(G_lattice_moore, target=ax, vertex_size=vertex_size)
ax.set_title(fr"Topology ($|\mathcal{{N}}|=8, N={N**2}$)")

fig.tight_layout()

In [ ]:
spacetime_diagrams=[]
T=N
for model in models:
    initial_config = np.random.randint(0,2,N**2)
    # initial_config = np.zeros(N**2)
    # initial_config[N**2//2+N//2] = 1
    H = model.forward(edges_lattice_moore, tc.tensor(initial_config[np.newaxis,:]), T=T)
    spacetime_diagram = np.array(H, dtype=int)[0].reshape(T+1,N,N)
    spacetime_diagrams.append(spacetime_diagram)

In [ ]:
SAVEGIF = False

# example_index = 7
for example_index in range(11):

    diagram = spacetime_diagrams[example_index]
    model = models[example_index]

    ani = make_animation_object(diagram, model)

    if SAVEGIF:
        # Save animations as GIFs
        loc = '../figures/gifs/'
        savename = f"animation_{model.__str__()}_on_lattice_moore_random_seed.gif"
        ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow

# show inline
# HTML(ani.to_jshtml())

# Time-evolution patterns on the original network

As a final step, we can look at the TEPs on the network that the perfomance was good in. Here we will need to present the dynamics in the same shape as a spacetime diagram, and we have to break ambiguity: how to order the nodes?

We will use PageRank to order te nodes. We will consider synthetic and metabolic networks

### _Load the data_

In [ ]:
#EE import classes from submodule

## fix path
import sys, os
# path to src
sys.path.append(os.path.abspath(".."))
# path to submodule
sys.path.append(os.path.abspath("../src/network_analysis"))

## get classes
from src.network_analysis.lib.datasources import SyntheticNetworks #, KeggMetabolicNetworks

In [ ]:
# load dataframe with networks
base_folder = "..\\src\\network_analysis\\data"
SyntheticNetworks.setup(base_folder=base_folder, random_seed=99999, loader_mode='full')

networks_scalefree = SyntheticNetworks.scalefree(samples_per_label=20, balanced=True)

In [ ]:
# load additional classes
import numpy as np
import torch as tc
import torch_geometric as tg
import torch.nn.functional as F

# from torch_sparse import SparseTensor
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch.utils.data import Dataset, DataLoader, Subset, RandomSampler, WeightedRandomSampler
from sklearn.model_selection import train_test_split

from typing import Union

#===========================================================
class DiscreteStateNetwork(Data):
    def __init__(self, edge_index:list=[[0], [0]], y:int=0, num_states:int=2, num_inits:int=1, *args, **kwargs):
        """
        `edge_index`: indices of the edges (i,j) in the form [[i1, ... im], [j1, ... jm]]
        `y`: integer code of the respective class
        `num_states`: the number of states to randomly generate the state vector (default: binary network)
        `num_inits`: the number of different initial configurations to generate, also affects the y vector (expansion)
        """
        num_nodes = np.max(edge_index) + 1
        super(DiscreteStateNetwork, self).__init__(
            edge_index = tc.tensor(edge_index, dtype=tc.long), # shape of [2, |E|]
            node_state = tc.randint(0, num_states, (num_nodes, num_inits)).float(), # shape of [|V|, L]
            y = tc.tensor(y).expand(num_inits), # shape of [L]
            batch = tc.zeros(num_nodes), # shape of [|V|]
            *args, **kwargs
        )

    def unpack(self):
        return tuple([self.edge_index, self.node_state.T, self.batch, self.y])


#===========================================================
class DiscreteStateNetworkDataset(Dataset):
    def __init__(self, network_collection, device:str='cpu', *args, **kwargs):
        classes, labels = np.unique(network_collection['label'], return_inverse=True)
        for G in network_collection['graph']:
            G.data = G.data.as_directed()
        self.data = [ 
            DiscreteStateNetwork(np.transpose(G.edges()), y, *args, **kwargs) 
            for (G, y) in zip(network_collection['graph'], labels)
        ]
        self.classes = tuple(classes)
        self.device = device

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx:int):
        return self.data[idx].to(self.device)

    @property
    def labels(self):
        return tc.stack([ G.y for G in self ], 0)

    def _random_loader(self, indices, num_samples:Union[int,float]=0, balance:bool=False, *args, **kwargs):
        subset  = Subset(self, indices)
        if isinstance(num_samples, float):
            num_samples = int(num_samples * len(subset))
        num_samples = min(num_samples, len(subset))
        if not num_samples:
            sampler = None
        elif not balance:
            sampler = RandomSampler(subset, False, num_samples)
        else:
            labels = self.labels[indices, 0]
            chances = 1.0 / np.bincount(labels)
            weights = chances[labels]
            sampler = WeightedRandomSampler(weights, num_samples, True)
        loader = GraphDataLoader(subset, sampler=sampler, *args, **kwargs)
        return loader

    def _index_split(self, pool:list, proportion:list, stratify:bool, seed:int=None):
        indices = []
        taken = 0.0
        for p in proportion:
            percent = p / (1.0 - taken)
            taken += p
            if taken >= 1.0:
                break
            elif percent == 0:
                indices.append([])
                continue
            strat = self.labels[pool].to('cpu') if stratify else None
            selected, pool = train_test_split(pool, train_size=percent, stratify=strat, random_state=seed)
            indices.append(selected)
        indices.append(pool)
        return indices

    def split(self, training:float, validation:float=0, stratify:bool=True, folds:int=0, pool:list=None, sampling:tuple=(0,0,0), 
                repeat:int=0, seed:int=None, *args, **kwargs):
        if 'batch_size' not in kwargs: # temporary, it should be in the named param list
            kwargs['batch_size'] = 1
        if repeat > 0:
            if seed is None:
                seed = [None]*repeat
            return [ 
                self.split(training, validation, stratify, folds, pool, sampling, repeat=0, seed=seed[i], *args, **kwargs) 
                for i in range(repeat) 
            ]
        if pool is None:
            pool = np.arange(len(self))
        if 'batch_size' not in kwargs:
            kwargs['batch_size'] = None
        if folds in [0, 1]:
            (tr_idx, vl_idx, te_idx) = self._index_split(pool, [training, validation], stratify, seed)
            loaders = {
                'train': self._random_loader(tr_idx, sampling[0], *args, **kwargs), 
                'valid': self._random_loader(vl_idx, sampling[1], *args, **kwargs), 
                'test': self._random_loader(te_idx, sampling[-1], *args, **kwargs) 
            }
            return loaders if not folds else [loaders]
        else:
            folds_idx = self._index_split(pool, [1.0/folds] * (folds-1), stratify, seed)
            dataloaders = []
            for te_idx in folds_idx:
                scale = 1 / (training + validation)
                subpool = list(set(pool) - set(te_idx))
                if validation > 0:
                    (tr_idx, vl_idx) = self._index_split(subpool, [training*scale, validation*scale], stratify, seed)
                else:
                    (tr_idx, vl_idx) = (subpool, [])
                dataloaders.append({
                    'train': self._random_loader(tr_idx, sampling[0], *args, **kwargs),
                    'valid': self._random_loader(vl_idx, sampling[1], *args, **kwargs), 
                    'test': self._random_loader(te_idx, sampling[-1], *args, **kwargs)
                })
            return dataloaders

In [ ]:
grouped = networks_scalefree.groupby('label')
keys = networks_scalefree.label.unique()
networks_dict = {key: group for key, group in grouped}

dataset_dict = {key: DiscreteStateNetworkDataset(networks_dict[key], num_states=2, num_inits=1) for key in keys}

In [ ]:
label = np.random.choice(keys)
print(f"Working with label {label} ...")

graphs_scalefree = []
edgess_scalefree = []
for G in dataset_dict[keys[0]]:    
    (E, h, _, y) = G.unpack()
    edges_scalefree = E.numpy().T
    N = h.shape[1]
    graph_scalefree = ig.Graph(n=N, edges=edges_scalefree)
    graphs_scalefree.append(graph_scalefree)
    edgess_scalefree.append(E)

### _Create TEPs_

In [ ]:
# NOTE the code below is ugly for sure

# select dataset
dataset = 'scalefree'
df_dataset = df[df['dataset']==dataset]

# select resolution and sort
df_dataset_R3 = df_dataset[df_dataset["resolution"]==3].sort_values(by='accuracy', ascending=False).reset_index(drop=True)
df_dataset_R4 = df_dataset[df_dataset["resolution"]==4].sort_values(by='accuracy', ascending=False).reset_index(drop=True)
df_dataset_R5 = df_dataset[df_dataset["resolution"]==5].sort_values(by='accuracy', ascending=False).reset_index(drop=True)

# find five best LLNAs per resolution
select_first_somany = 9

born_if_list_R3 = df_dataset_R3['binary sequence B'][:select_first_somany]
born_if_list_R4 = df_dataset_R4['binary sequence B'][:select_first_somany]
born_if_list_R5 = df_dataset_R5['binary sequence B'][:select_first_somany]

survive_if_list_R3 = df_dataset_R3['binary sequence S'][:select_first_somany]
survive_if_list_R4 = df_dataset_R4['binary sequence S'][:select_first_somany]
survive_if_list_R5 = df_dataset_R5['binary sequence S'][:select_first_somany]

born_if_list_R3 = [[i for i, x in enumerate(born_if) if x] for born_if in born_if_list_R3]
born_if_list_R4 = [[i for i, x in enumerate(born_if) if x] for born_if in born_if_list_R4]
born_if_list_R5 = [[i for i, x in enumerate(born_if) if x] for born_if in born_if_list_R5]

survive_if_list_R3 = [[i for i, x in enumerate(survive_if) if x] for survive_if in survive_if_list_R3]
survive_if_list_R4 = [[i for i, x in enumerate(survive_if) if x] for survive_if in survive_if_list_R4]
survive_if_list_R5 = [[i for i, x in enumerate(survive_if) if x] for survive_if in survive_if_list_R5]

models_R3 = [LLNA(3, x=born_if, y=survive_if) for born_if, survive_if in zip(born_if_list_R3, survive_if_list_R3)]
models_R4 = [LLNA(4, x=born_if, y=survive_if) for born_if, survive_if in zip(born_if_list_R4, survive_if_list_R4)]
models_R5 = [LLNA(5, x=born_if, y=survive_if) for born_if, survive_if in zip(born_if_list_R5, survive_if_list_R5)]

In [ ]:
TEPs_R3=[]
TEPs_R4=[]
TEPs_R5=[]
T=N//2
for model_R3, model_R4, model_R5, graph_scalefree, edges_scalefree \
    in zip(models_R3, models_R4, models_R5, graphs_scalefree[:select_first_somany], edgess_scalefree[:select_first_somany]):
    # find pagerank
    pagerank_order = np.argsort(graph_scalefree.pagerank())
    print(f"{model_R3.__str__()}, {model_R4.__str__()}, {model_R5.__str__()}")
    # same initial condition in each case
    initial_config = np.random.randint(0,2,N)
    H_R3 = model_R3.forward(edges_scalefree, tc.tensor(initial_config[np.newaxis,:]), T=T)
    H_R4 = model_R4.forward(edges_scalefree, tc.tensor(initial_config[np.newaxis,:]), T=T)
    H_R5 = model_R5.forward(edges_scalefree, tc.tensor(initial_config[np.newaxis,:]), T=T)
    # create TEP
    TEP_R3 = np.array(H_R3, dtype=int)[0]
    TEP_R4 = np.array(H_R4, dtype=int)[0]
    TEP_R5 = np.array(H_R5, dtype=int)[0]
    # sort TEP along Pagerank
    TEP_R3 = TEP_R3[:,pagerank_order]
    TEP_R4 = TEP_R4[:,pagerank_order]
    TEP_R5 = TEP_R5[:,pagerank_order]
    # append to list
    TEPs_R3.append(TEP_R3)
    TEPs_R4.append(TEP_R4)
    TEPs_R5.append(TEP_R5)

In [ ]:
# NOTE: hardcoded for now
fig, axs = plt.subplots(3,3,figsize=(14,8))
for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        idx = i*3+j
        ax.imshow(TEPs_R3[idx], cmap='Greys')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(fr"$\rightarrow$ Descending PageRank $\rightarrow$")
        ax.set_title(fr"{models_R3[idx].__str__(latex=True)}")

fig.suptitle(f"Scalefree network with label '{label}': various high-accuracy rules with Resolution $3$", fontsize=16)

In [ ]:
# NOTE: hardcoded for now
fig, axs = plt.subplots(3,3,figsize=(14,8))
for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        idx = i*3+j
        ax.imshow(TEPs_R4[idx], cmap='Greys')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(fr"$\rightarrow$ Descending PageRank $\rightarrow$")
        ax.set_title(fr"{models_R4[idx].__str__(latex=True)}")

fig.suptitle(f"Scalefree network with label '{label}': various high-accuracy rules with Resolution $4$", fontsize=16)

In [ ]:
# NOTE: hardcoded for now
fig, axs = plt.subplots(3,3,figsize=(14,8))
for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        idx = i*3+j
        ax.imshow(TEPs_R5[idx], cmap='Greys')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(fr"$\rightarrow$ Descending PageRank $\rightarrow$")
        ax.set_title(fr"{models_R5[idx].__str__(latex=True)}")

fig.suptitle(f"Scalefree network with label '{label}': various high-accuracy rules with Resolution $5$", fontsize=16)

# Time-evolution pattern on a different (smaller) scale-free network

In order to better see what's going on in a detailed fashion, let's do the same but with a smaller network.

In [ ]:
# for illustration

N = 50#**2
min_edges = 2
G_scalefree = ig.Graph.Barabasi(N, min_edges)

A = np.array([ *G_scalefree.get_adjacency() ])

fig, axs = plt.subplots(1,3,figsize=(12,4))
ig.plot(G_scalefree, target=axs[0], vertex_size=10)

axs[1].imshow(A, cmap='Greys')
axs[1].set_title("Adjacency matrix")

degrees = G_scalefree.degree()
min_degree = min(degrees); max_degree = max(degrees)
bins = np.arange(min_degree, max_degree+2)-.5

axs[2].hist(degrees, bins=bins)
axs[2].set_xlim(min_degree-1.5, max_degree+1.5)
axs[2].set_xticks(bins[:-1]+.5)
axs[2].set_title("Degree distribution")

fig.tight_layout()

In [ ]:
# for real

N = 100
min_edges = 2
G_scalefree = ig.Graph.Barabasi(N, min_edges)

G_scalefree.to_directed()
edges_scalefree = tc.tensor(G_scalefree.get_edgelist()).T
G_scalefree.to_undirected()

pagerank_order = np.argsort(G_scalefree.pagerank())

TEPs_R3=[]
TEPs_R4=[]
TEPs_R5=[]
T=N//2
for model_R3, model_R4, model_R5 in zip(models_R3, models_R4, models_R5):#, total=select_first_somany):
    print(f"{model_R3.__str__()}, {model_R4.__str__()}, {model_R5.__str__()}")
    # same initial condition in each case
    initial_config = np.random.randint(0,2,N)
    H_R3 = model_R3.forward(edges_scalefree, tc.tensor(initial_config[np.newaxis,:]), T=T)
    H_R4 = model_R4.forward(edges_scalefree, tc.tensor(initial_config[np.newaxis,:]), T=T)
    H_R5 = model_R5.forward(edges_scalefree, tc.tensor(initial_config[np.newaxis,:]), T=T)
    # create TEP
    TEP_R3 = np.array(H_R3, dtype=int)[0]
    TEP_R4 = np.array(H_R4, dtype=int)[0]
    TEP_R5 = np.array(H_R5, dtype=int)[0]
    # sort TEP along Pagerank
    TEP_R3 = TEP_R3[:,pagerank_order]
    TEP_R4 = TEP_R4[:,pagerank_order]
    TEP_R5 = TEP_R5[:,pagerank_order]
    # append to list
    TEPs_R3.append(TEP_R3)
    TEPs_R4.append(TEP_R4)
    TEPs_R5.append(TEP_R5)

In [ ]:
# NOTE: hardcoded for now
fig, axs = plt.subplots(3,3,figsize=(12,7))
for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        idx = i*3+j
        ax.imshow(TEPs_R3[idx], cmap='Greys')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(fr"$\rightarrow$ Descending PageRank $\rightarrow$")
        ax.set_title(fr"{models_R3[idx].__str__(latex=True)}")

fig.suptitle("Scalefree network: various high-accuracy rules with Resolution $3$", fontsize=16)

In [ ]:
# NOTE: hardcoded for now
fig, axs = plt.subplots(3,3,figsize=(12,7))
for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        idx = i*3+j
        ax.imshow(TEPs_R4[idx], cmap='Greys')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(fr"$\rightarrow$ Descending PageRank $\rightarrow$")
        ax.set_title(fr"{models_R4[idx].__str__(latex=True)}")

fig.suptitle("Scalefree network: various high-accuracy rules with Resolution $4$", fontsize=16)

In [ ]:
# NOTE: hardcoded for now
fig, axs = plt.subplots(3,3,figsize=(12,7))
for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        idx = i*3+j
        ax.imshow(TEPs_R5[idx], cmap='Greys')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(fr"$\rightarrow$ Descending PageRank $\rightarrow$")
        ax.set_title(fr"{models_R5[idx].__str__(latex=True)}")

fig.suptitle("Scalefree network: various high-accuracy rules with Resolution $5$", fontsize=16)

It's not entirely clear what we should take away from this. What patterns do we see? This will require some more statistical analysis and definitely justifies a non-supervised classification approach (much like texture analysis).